# Lab 01 — Your first agentic workflow

**Module:** [01 — Foundations & the evals mindset](https://codexplorerepo.github.io/agentic-ai/modules/01-foundations/)

You are going to build a **video research brief agent**: give it a topic, get
back a brief you could script a video from — hook, key points, what's actually
new, and sources.

Then you will build the thing it has to beat: one big prompt, no tools.

Then you will find out which one is better, on twelve cases, with numbers.

That last part is the point of this lab. Anyone can build an agent. Knowing
whether it was worth building is the skill.

---

### What this costs

A few cents. A `BudgetGuard` is active throughout and will **raise** rather
than warn if a run overspends.

### What you need

- One model provider key (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, or `GOOGLE_API_KEY`)
- A `TAVILY_API_KEY` for web search — [free tier](https://tavily.com) is plenty

See [setup](https://codexplorerepo.github.io/agentic-ai/start-here/setup) if
anything below fails to import.

## 0. Check your environment

If this cell prints no providers, your `.env` is not being read. It must be at
the repo root, and if you edited it after starting this kernel, run
`get_settings.cache_clear()`.

In [ ]:
from agentic_ai import get_settings
from agentic_ai.llm import available_providers

settings = get_settings()
print("model providers ready:", available_providers() or "NONE — check .env")
print("web search ready:    ", settings.has_key("tavily"))
print("run budget:           ${:.2f}".format(settings.agentic_run_budget_usd))

## 1. The baseline, first

We are building the *simplest thing that could work* before the interesting
thing. This ordering matters more than it sounds: it is how you find out
whether the agent you were about to build is actually better.

Both systems live in [`systems.py`](./systems.py) rather than in these cells,
so that this notebook and `run_evals.py` run **exactly the same code**. A
system defined twice is a system whose eval numbers mean nothing.

Read the two system prompts there now — they are deliberately identical apart
from the tool instructions.

In [ ]:
import sys
sys.path.insert(0, ".")

from systems import BRIEF_SPEC, research_agent, single_prompt

print(BRIEF_SPEC)

### Run the baseline

Pick a topic that needs *current* information — that is where the baseline
should struggle.

In [ ]:
TOPIC = "What changed in agentic AI tooling in the last six months?"

baseline_brief = single_prompt(TOPIC)
print(baseline_brief)

**Look at what it did.** Typically one of two things:

- It hedged honestly: "I don't have access to recent information."
- It answered confidently from training data, with no sources, possibly stale.

The second is the dangerous one. It *reads* like a good brief. Nothing about
the output signals that the facts might be a year old. This is the failure mode
the whole module is about: agents and LLMs fail by producing plausible output,
not by crashing.

## 2. The agent

Same task. Now the model gets two tools and decides for itself how to use them.

Everything about the loop — step limit, budget, trace, error recovery — comes
from [`agentic_ai.patterns.run_agent`](../../src/agentic_ai/patterns/tool_use.py).
Read that file at some point today; it is about eighty lines and it is what
sits underneath every agent framework you will ever evaluate.

In [ ]:
from agentic_ai.llm import BudgetGuard
from agentic_ai.patterns import run_agent
from agentic_ai.tools import Toolbox
from agentic_ai.tools.fetch import fetch_page
from agentic_ai.tools.web_search import search_web

from systems import AGENTIC_SYSTEM

toolbox = Toolbox(search_web, fetch_page)

# This is what the model actually sees about each tool. The description is
# prompt, not documentation -- it is the only thing telling the model when
# search_web is the right choice rather than fetch_page.
for spec in toolbox.specs:
    print(f"=== {spec.name} ===")
    print(spec.description.strip()[:300])
    print("args:", list(spec.parameters["properties"]), "\n")

### Watch it work

`on_step` lets us print the model's decision at each step instead of staring at
a blank cell. This is also the cheapest form of observability there is.

In [ ]:
def show_step(step, response):
    if response.wants_tools:
        calls = ", ".join(
            f"{c.name}({list(c.arguments.values())[0]!r:.60})" if c.arguments else c.name
            for c in response.tool_calls
        )
        print(f"  step {step}: -> {calls}")
    else:
        print(f"  step {step}: answered ({len(response.text):,} chars)")


result = run_agent(
    f"Research topic: {TOPIC}",
    system=AGENTIC_SYSTEM,
    toolbox=toolbox,
    max_steps=10,
    budget=BudgetGuard(budget_usd=0.30),
    on_step=show_step,
)

print()
print(result.summary())
print("tools used:", result.tool_calls_made)

In [ ]:
print(result.output)

### Always check `succeeded`

`run_agent` returns text even when it hit the step limit — and that text is the
model's last unfinished thought, not an answer. Code that treats the two the
same will happily ship a fragment.

In [ ]:
print("stopped because:", result.stopped_because)
print("succeeded:       ", result.succeeded)
print("steps:           ", result.steps)
print("cost:            ${:.4f}".format(result.cost_usd))

## 3. Read the trace

The output does not tell you that the agent searched four times, got nothing
useful twice, and gave up. The trace does.

This is the artefact that makes an agent failure reproducible, and the reason
`run_agent` records one whether you asked for it or not.

In [ ]:
for span in result.trace.spans:
    indent = "  " if span.parent_id else ""
    detail = ""
    if span.kind == "tool":
        q = span.attributes.get("arguments", {})
        detail = f"  {str(list(q.values())[0] if q else '')[:60]!r}"
        if span.attributes.get("looks_like_error"):
            detail += "  <-- ERROR"
    elif span.kind == "llm":
        detail = f"  {span.attributes.get('tokens', 0):,} tokens"
    print(f"{indent}{span.kind:5s} {span.name:14s} {span.duration_s:6.2f}s{detail}")

print()
print(result.trace.summary())

## 4. Now measure it

Two outputs, side by side, and the agent's looks better. That is not evidence —
it is one case, chosen by us, judged by us, on the dimension we happened to
notice.

Twelve cases, scored consistently, is evidence.

The eval set is
[`evals/video_brief_cases.yaml`](./evals/video_brief_cases.yaml). Open it. It is
tagged by kind — `current`, `definition`, `comparison`, `obscure`, `easy` —
because the useful question is never "what is the overall score" but "which
*kind* of request does this fail on".

In [ ]:
from agentic_ai.evals import EvalDataset

dataset = EvalDataset.from_yaml("evals/video_brief_cases.yaml")

print(f"{len(dataset)} cases")
for tag in ("current", "definition", "comparison", "obscure", "easy"):
    print(f"  {tag:11s} {len(dataset.filter_by_tag(tag))}")

### A free check before any paid one

Deterministic checks cost nothing and are never wrong about what they measure.
Run them first, always.

This one counts distinct cited URLs — because the failure this lab most wants
to surface is confident prose with nothing behind it, and a judge is not needed
to spot it.

In [ ]:
from run_evals import cites_sources

for name, text in (("baseline", baseline_brief), ("agentic", result.output)):
    score = cites_sources(text, dataset.cases[0])[0]
    print(f"{name:9s} cites_sources={score.value:.0f}  {score.reason}")

### The full comparison

This spends a few cents per system: each case runs the system, then a judge
call scores it against the case's rubric.

Run it from the terminal so you can watch it, and so the scorecards land next
to the eval set:

```bash
make eval-01
```

Or, for a free pass that skips the judge and only checks the deterministic
parts:

```bash
uv run python labs/01-foundations/run_evals.py --no-judge
```

In [ ]:
# Uncomment to run inside the notebook instead of the terminal.
# Judge off by default here so an accidental run costs nothing.

# from agentic_ai.evals import compare, evaluate
#
# baseline_report = evaluate(
#     single_prompt, dataset, system_name="single-prompt",
#     use_judge=False, extra_checks=[cites_sources],
# )
# agentic_report = evaluate(
#     research_agent, dataset, system_name="agentic",
#     use_judge=False, extra_checks=[cites_sources],
# )
# print(compare(baseline_report, agentic_report))

### Reading the result honestly

Three things to hold onto:

**A one-case swing on twelve cases is noise.** `compare()` refuses to call
differences within about one case, because the fastest way to waste a week is
chasing a number that moved because of sampling.

**Look at the per-tag breakdown, not the total.** The agent usually wins
decisively on `current` and `obscure`, and ties on `definition`. That pattern
is a finding: it says *route* — send definitional questions to the cheap single
prompt and only pay for the agent when the question needs fresh information.

**If the agent did not win, that is a result.** It means this task did not need
agency, and you just saved yourself from shipping a slower, more expensive
system. See
[when not to use agents](https://codexplorerepo.github.io/agentic-ai/foundations/when-not-to-use-agents).

## 5. Break it on purpose

Three deliberate failures. This is the part you will remember.

### Failure 1 — a tool that always fails, and no step limit

Without a step limit, a tool the model cannot satisfy produces a loop that
bills you the whole way. Watch the budget guard stop what the step limit
would have.

In [ ]:
from agentic_ai.errors import BudgetExceeded
from agentic_ai.tools import tool


@tool
def broken_search(query: str) -> str:
    """Search the web for information.

    Args:
        query: What to search for.
    """
    raise RuntimeError("search backend unavailable")


try:
    doomed = run_agent(
        "Research topic: anything at all",
        system=AGENTIC_SYSTEM,
        toolbox=Toolbox(broken_search),
        max_steps=50,                      # deliberately far too high
        budget=BudgetGuard(budget_usd=0.05),  # the thing that saves us
    )
    print(doomed.summary())
    print("stopped because:", doomed.stopped_because)
except BudgetExceeded as exc:
    print("BudgetExceeded — the guard did its job")
    print(exc)

Note what the model did: it was *told* the tool failed, as text, and it
kept trying — reasonably, since a transient failure is worth one retry. The
model cannot know the tool is permanently broken. That is your job, which is
why the step limit and the budget are **your** responsibility and not the
model's.

### Failure 2 — errors the model never sees

Our `Toolbox.dispatch` returns tool errors as *text* so the model can recover.
Here is the same run with the error raised instead.

In [ ]:
messages_seen = [m for m in doomed.messages if m.role == "tool"] if "doomed" in dir() else []
if messages_seen:
    print("What the model was told about the failure:\n")
    print(messages_seen[0].content)
    print(
        "\nThis is why recovery is possible. Raise instead of returning this,\n"
        "and the run dies with the model never learning anything."
    )

### Failure 3 — a topic with no good sources

The `obscure` cases exist for this. A well-behaved system says it could not
find reliable data. A badly-behaved one invents a statistic, and **no check
except the rubric catches it** — which is exactly why rubric judging earns its
cost.

In [ ]:
obscure_case = dataset.filter_by_tag("obscure").cases[0]
print("Case:", obscure_case.id)
print(obscure_case.input.strip())
print("\nRubric:", obscure_case.rubric.strip()[:300])

obscure_output = research_agent(obscure_case.input)
print("\n--- output ---\n")
print(obscure_output[:1500])

In [ ]:
# Did it invent numbers? Look for suspiciously specific figures.
import re

figures = re.findall(r"\b\d+(?:\.\d+)?%|\b\d{1,3},\d{3}\b", obscure_output)
admits = any(
    p in obscure_output.lower()
    for p in ("could not find", "no reliable", "unable to find", "not find reliable")
)

print("specific figures found:", figures or "none")
print("admits thin sourcing:  ", admits)
print()
print("GOOD" if admits else "SUSPECT — check every figure above against the cited source")

## What to take away

1. **Agency is a control-flow decision.** You chose to let the model pick the
   path. That buys adaptability and costs predictability.
2. **Build the baseline first.** Sometimes it wins, and finding that out in an
   hour instead of a fortnight is the whole return on this practice.
3. **The eval set is the deliverable.** Without it you cannot tell whether
   anything you do next helps.
4. **A step limit and a spend ceiling are correctness features.** The model
   cannot tell that a tool is permanently broken. You can.
5. **Tool errors belong in the conversation, not in a traceback.** A model told
   why its call failed usually fixes it.
6. **Read the trace, not the output.** The output never admits what the agent
   actually did.

## Exercises

In [`exercises/`](./exercises/) — each is graded by adding cases to the eval
set and watching the score, not by a hidden test:

1. **Add a third system** — a fixed chain (generate questions → search each →
   synthesise), with no agency at all. Does it beat both?
2. **Make the agent cheaper** without losing pass rate. Try a smaller model, a
   lower `max_steps`, or fewer search results. Measure each.
3. **Add a check that catches a failure you saw** in your own run, and confirm
   it fails on the output that prompted it.

## Next

[Module 02 — Reflection](https://codexplorerepo.github.io/agentic-ai/modules/02-reflection/):
make the system critique its own output, and find out whether draft two is
actually better than draft one.